In [ ]:
import requests
import time
import os
import csv

from dotenv import load_dotenv

load_dotenv("../src/config/.env")

API_TOKEN = os.getenv("GITHUB_TOKEN")

LANGUAGES = ['C', 'C++', 'C#', 'Java', 'JavaScript', 'TypeScript', 'Python']

TARGET_COUNT_PER_LANG = 400

# Minimum repository size in Kilobytes (10 MB = 10,000 KB)
MIN_SIZE_KB = 10000

BLACKLISTED_REPOS = {
    'google', 'microsoft', 'facebook', 'apple', 'amazon', 'netflix', 'ibm',
    'oracle', 'intel', 'adobe', 'airbnb', 'uber', 'linkedin', 'twitter',
    'mozilla', 'apache', 'torvalds', 'docker', 'kubernetes', 'tensorflow',
    'pytorch', 'angular', 'vuejs', 'reactjs', 'nodejs', 'golang', 'rust-lang',
    'jetbrains', 'elastic', 'mongodb', 'automattic', 'square', 'shopify',
    'stripe', 'spotify', 'dropbox', 'github', 'gitlab', 'atlassian', 'slack',
    'zoom', 'salesforce', 'unity', 'unreal', 'epic', 'valve', 'steam',
    'alphabet', 'samsung', 'samsungelectronics', 'honhai', 'honhaiprecision',
    'meta', 'metaplatforms', 'huawei', 'huaweiinvestment', 'sony', 'dell',
    'delltechnologies', 'tencent', 'tencentholdings', 'taiwan', 'taiwansemiconductor',
    'tsmc', 'hitachi', 'lg', 'lgelectronics', 'accenture', 'nvidia', 'panasonic',
    'panasonicholdings', 'cisco', 'ciscosystems', 'lenovo', 'lenovogroup',
    'hp', 'pegatron', 'xiaomi', 'ubertech', 'ubertechnologies', 'qualcomm',
    'broadcom', 'chinaelectronics', 'quanta', 'quantacomputer', 'jabil',
    'sap', 'sapse', 'luxshare', 'luxshareprecision', 'mozilla', 'apache', 'gnu',
    'gnulinux', 'linuxfoundation', 'oniro', 'railcasts', 'cloudnative', 'cncf',
    'gnome', 'kde', 'openstack', 'osgeo', 'opensourcegeospatial', 'softwareheritage',
    'openknowledge', 'wikimedia', 'ourresearch', 'berkeley', 'mit', 'stanford',
    'stanforduniversity', 'audiopedia', 'audiopediafoundation', 'opensourcesecurity',
    'openssf', 'openjs', 'openjsfoundation', 'academysoftware', 'academysoftwarefoundation',
    'openmobility', 'openmobilityfoundation', 'osu', 'opensource', 'fossi',
    'fossifoundation', 'openwallet', 'openwalletfoundation', 'verapdf', 'nomic',
    'nomicfoundation', 'farama', 'faramafoundation', 'fintech', 'fintechopensource',
    'communityox', 'commonhaus'
}

GITHUB_API_URL = "https://api.github.com/search/repositories"
#API_TOKEN = ''

#Check if API is valid
def check_prerequisites():
    if not API_TOKEN:
        print("ERROR: GitHub API token not found.")
        print("Please set the GITHUB_TOKEN environment variable.")
        exit(1)

def fetch_repos_for_language(language, headers):

    print(f"\n--- Starting search for language: {language} ---")
    found_repos = []
    
 
    star_ranges = ['>10000', '5000..9999', '2000..4999', '1000..1999', '500..999', '250..499', '100..249']

    exclusions_str = " ".join([f"-user:{owner}" for owner in BLACKLISTED_REPOS])

    for star_range in star_ranges:
        if len(found_repos) >= TARGET_COUNT_PER_LANG:
            break

        print(f"\n[{language}] Searching with star range: {star_range}")
        page = 1
        
        while page <= 10:
            if len(found_repos) >= TARGET_COUNT_PER_LANG:
                break
                
            query = f'language:"{language}" size:>={MIN_SIZE_KB} stars:{star_range} {exclusions_str}'
            
            params = {
                'q': query,
                'sort': 'stars',
                'order': 'desc',
                'per_page': 100,
                'page': page
            }
            
            # Retry loop with exponential backoff ###
            retry_attempts = 5
            base_wait_time = 10  # Start with 10 seconds

            for attempt in range(retry_attempts):
                try:
                    # Be more polite to the API with a longer delay
                    time.sleep(2) 
                    
                    response = requests.get(GITHUB_API_URL, headers=headers, params=params)
                    
                    # Handle 403 Secondary Rate Limit
                    if response.status_code == 403:
                        response_json = response.json()
                        if "secondary rate limit" in response_json.get("message", ""):
                            wait_time = base_wait_time * (2 ** attempt) # Exponential backoff
                            print(f"   [!] Hit secondary rate limit. Waiting for {wait_time} seconds before retrying...")
                            time.sleep(wait_time)
                            continue # Retry the same request
                    
                    # Handle 422 (1000 result limit) by moving to the next star range
                    if response.status_code == 422:
                        print(f"   [*] Reached API limit for star range '{star_range}'. Moving to next range.")
                        page = 999 # Break out of the page loop
                        break

                    response.raise_for_status() # Handle other HTTP errors
                    
                    # If successful, break the retry loop
                    data = response.json()
                    break 

                except requests.exceptions.RequestException as e:
                    print(f"   [!] A network error occurred: {e}")
                    wait_time = base_wait_time * (2 ** attempt)
                    print(f"   [!] Waiting for {wait_time} seconds before retrying network request...")
                    time.sleep(wait_time)
            else: # This 'else' belongs to the 'for' loop, runs if the loop completes without 'break'
                print(f"   [!] Failed to fetch data after {retry_attempts} attempts. Skipping this page.")
                data = {} 
                page += 1 
                continue

            items = data.get('items', [])
            
            if not items:
                print(f"   [*] No more results in this star range. Moving on.")
                break 

            for repo in items:
                owner_type = repo['owner']['type']
                owner_login = repo['owner']['login'].lower()

                if owner_type == 'User' and owner_login not in BLACKLISTED_REPOS:
                    if not any(r['name'] == repo['full_name'] for r in found_repos):
                        found_repos.append({
                            'language': language,
                            'name': repo['full_name'],
                            'owner': repo['owner']['login'],
                            'stars': repo['stargazers_count'],
                            'forks': repo['forks_count'],
                            'size_kb': repo['size'],
                            'url': repo['html_url'],
                        })
                        print(f"[{language}] Found {len(found_repos)}/{TARGET_COUNT_PER_LANG}: {repo['full_name']}")

                        if len(found_repos) >= TARGET_COUNT_PER_LANG:
                            break
            
            page += 1

    print(f"--- Finished search for {language}. Found {len(found_repos)} matching repositories. ---")
    return found_repos


def save_to_csv(repositories, filename="individual_repositoriesV2.csv"):
    """Saves the list of repository data to a CSV file."""
    if not repositories:
        print("No repositories found to save.")
        return

    print(f"\nSaving {len(repositories)} repositories to {filename}...")
    
    # Columns for the CSV file
    fieldnames = [
        'language', 'name', 'owner', 'stars', 'forks', 'size_kb', 'url', 
    ]
    
    with open(filename, mode='w', newline='', encoding='utf-8') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(repositories)

    print(f"Successfully saved data to {filename}")

def main():
    """Main function to orchestrate the fetching and saving."""
    check_prerequisites()
    
    headers = {
        'Authorization': f'token {API_TOKEN}',
        'Accept': 'application/vnd.github.v3+json'
    }
    
    all_found_repos = []
    for lang in LANGUAGES:
        repos_for_lang = fetch_repos_for_language(lang, headers)
        all_found_repos.extend(repos_for_lang)

    save_to_csv(all_found_repos)

if __name__ == "__main__":
    main()